In [1]:
import sys, torch, pandas as pd, sklearn
print("python:", sys.version)
print("torch:", torch.__version__)
print("mps available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)

python: 3.11.14 (main, Oct 21 2025, 18:27:30) [Clang 20.1.8 ]
torch: 2.10.0
mps available: True
pandas: 3.0.1
sklearn: 1.8.0


In [2]:
df = pd.read_csv('/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-alladults.csv')

print("Number of samples:", len(df))
df.head()

Number of samples: 445834


,race,gender,age_at_visit,temperature,heartrate,resprate,sbp,dbp,o2sat,pain,unable,chiefcomplaint,acuity
0,Black/African American Only,0,30.0,98.3,128.0,16.0,149.0,79.0,100.0,10.0,0,"Carbuncle, furuncle, boil, cellulitis...",3
1,Black/African American Only,0,64.0,98.0,106.0,18.0,142.0,111.0,100.0,4.0,0,Painful urination,3
2,Blank,1,73.0,101.4,93.0,NaN,112.0,49.0,97.0,0.0,0,Other symptoms referable to the respi...,4
3,Black/African American Only,0,69.0,98.4,107.0,18.0,101.0,63.0,100.0,NaN,1,Other symptoms referable to the respi...,5
4,Blank,0,51.0,97.9,74.0,18.0,114.0,84.0,97.0,7.0,0,"Abdominal pain, cramps, spasms, NOS",3


In [ ]:
selected_cols = ["chiefcomplaint", "acuity"]
print("Columns:", list(df.columns))
df = df[selected_cols].copy()

# some basic cleaning
print("Rows before cleaning:", len(df))
df["chiefcomplaint"] = df["chiefcomplaint"].astype(str).fillna("").str.strip() 
df = df[df["chiefcomplaint"].str.len() > 0].copy()  # only keep examples with a chief complaint

print("Rows after cleaning 'chiefcomplaint':", len(df))

df = df[df["acuity"].isin([1,2,3,4,5])].copy()
df["acuity"] = df["acuity"].astype(int)
print("Rows after cleaning 'acuity':", len(df))

print("Rows after cleaning:", len(df))
print("\nAcuity distribution (counts):")
print(df["acuity"].value_counts().sort_index())
print("\nAcuity distribution (proportion):")
print((df["acuity"].value_counts(normalize=True).sort_index()).round(4))

# Text length stats (characters + approx words)
lens = df["chiefcomplaint"].str.len()
words = df["chiefcomplaint"].str.split().map(len)

print("\nChief complaint length (chars):")
print(lens.describe(percentiles=[.5,.9,.95,.99]).round(2))
print("\nChief complaint length (words):")
print(words.describe(percentiles=[.5,.9,.95,.99]).round(2))

# print("\nUnique subjects:", df["subject_id"].nunique())

Columns: ['race', 'gender', 'age_at_visit', 'temperature', 'heartrate', 'resprate', 'sbp', 'dbp', 'o2sat', 'pain', 'unable', 'chiefcomplaint', 'acuity']
Rows before cleaning: 445834
Rows after cleaning 'chiefcomplaint': 445817
Rows after cleaning 'acuity': 445817
Rows after cleaning: 445817

Acuity distribution (counts):
acuity
1     24042
2    143264
3    239670
4     36619
5      2222
Name: count, dtype: int64

Acuity distribution (proportion):
acuity
1    0.0539
2    0.3214
3    0.5376
4    0.0821
5    0.0050
Name: proportion, dtype: float64

Chief complaint length (chars):
count    445817.00
mean         15.28
std           8.76
min           1.00
50%          13.00
90%          28.00
95%          33.00
99%          40.00
max         136.00
Name: chiefcomplaint, dtype: float64

Chief complaint length (words):
count    445817.00
mean          2.51
std           1.30
min           1.00
50%           2.00
90%           4.00
95%           5.00
99%           6.00
max          16.00
Name

In [ ]:
from sklearn.model_selection import train_test_split

# note: keys 1-5 map to 0-4, which is 5 classes
label2id = {1:0, 2:1, 3:2, 4:3, 5:4}
id2label = {v:k for k,v in label2id.items()}
df["label"] = df["acuity"].map(label2id)

# standard random split: 80/10/10 (train/val/test)
train_df, test_df = train_test_split(df, test_size=0.10, random_state=42)

# reset indices
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# 0.1111111 of the remaining 90% ≈ 10% of total data
tr_df, val_df = train_test_split(train_df, test_size=0.1111111, random_state=43)  

# reset indices again
tr_df  = tr_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

def summarize_split(name, d):
    print(f"\n{name}: rows={len(d):,}")
    print(d["acuity"].value_counts(normalize=True).sort_index().round(4))

summarize_split("TRAIN", tr_df)
summarize_split("VAL", val_df)
summarize_split("TEST", test_df)

print("\n✅ Splitting complete. All instances treated independently.")


TRAIN: rows=356,653
acuity
1    0.0541
2    0.3215
3    0.5374
4    0.0820
5    0.0050
Name: proportion, dtype: float64

VAL: rows=44,582
acuity
1    0.0543
2    0.3185
3    0.5399
4    0.0819
5    0.0053
Name: proportion, dtype: float64

TEST: rows=44,582
acuity
1    0.0518
2    0.3228
3    0.5372
4    0.0834
5    0.0048
Name: proportion, dtype: float64

✅ Splitting complete. All instances treated independently.


In [8]:
# test_df
test_df.to_csv('/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/PythonFolder/clinical_test_df.csv', index=False)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score
import numpy as np

# build pipeline
tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        min_df=5,
        max_features=200_000
    )),
    ("clf", LogisticRegression(
        max_iter=300,
        n_jobs=-1,
        class_weight="balanced"   # IMPORTANT for imbalance
    ))
])

# train
tfidf_clf.fit(tr_df["chiefcomplaint"], tr_df["label"])

# validation predictions
val_pred = tfidf_clf.predict(val_df["chiefcomplaint"])

print("Validation Macro-F1:",
      f1_score(val_df["label"], val_pred, average="macro"))

print("Validation Balanced Accuracy:",
      balanced_accuracy_score(val_df["label"], val_pred))

print("\nPer-class performance:")
print(classification_report(val_df["label"], val_pred, digits=3))

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Validation Macro-F1: 0.3773393115248662
Validation Balanced Accuracy: 0.5625430697978299

Per-class performance:
              precision    recall  f1-score   support

           0      0.202     0.673     0.311      2422
           1      0.618     0.464     0.530     14201
           2      0.776     0.468     0.584     24071
           3      0.278     0.612     0.382      3653
           4      0.042     0.596     0.079       235

    accuracy                          0.490     44582
   macro avg      0.383     0.563     0.377     44582
weighted avg      0.650     0.490     0.533     44582



In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import evaluate

model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    # complaints are tiny; 32 is plenty
    return tokenizer(batch["chiefcomplaint"], truncation=True, max_length=32)

# build HF datasets
train_ds = Dataset.from_pandas(tr_df[["chiefcomplaint", "label"]])
val_ds   = Dataset.from_pandas(val_df[["chiefcomplaint", "label"]])
test_ds  = Dataset.from_pandas(test_df[["chiefcomplaint", "label"]])

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])
val_ds   = val_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])
test_ds  = test_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# model: 5 labels (0-4)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

args = TrainingArguments(
    output_dir="triage_text_bioclinicalbert",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    logging_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to="none",
    fp16=False,
    bf16=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Map:   0%|          | 0/356653 [00:00<?, ? examples/s]

Map:   0%|          | 0/44582 [00:00<?, ? examples/s]

Map:   0%|          | 0/44582 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

...

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=22292, training_loss=0.7962916514699386, metrics={'train_runtime': 6173.543, 'train_samples_per_second': 115.542, 'train_steps_per_second': 3.611, 'total_flos': 6197231887507032.0, 'train_loss': 0.7962916514699386, 'epoch': 2.0})

In [11]:
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score, confusion_matrix
import numpy as np

# Get predictions
test_output = trainer.predict(test_ds)

test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids

print("TEST RESULTS")
print("Macro-F1:", f1_score(test_labels, test_preds, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(test_labels, test_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, digits=3))

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TEST RESULTS
Macro-F1: 0.43344653479362993
Balanced Accuracy: 0.40997048254561286

Confusion Matrix:
[[  572  1068   661     9     0]
 [  513  8625  5120   129     2]
 [  114  3854 18486  1489     7]
 [    4    90  2167  1447     9]
 [    0     9    95   103     9]]

Classification Report:
              precision    recall  f1-score   support

           0      0.475     0.248     0.326      2310
           1      0.632     0.599     0.615     14389
           2      0.697     0.772     0.732     23950
           3      0.455     0.389     0.420      3717
           4      0.333     0.042     0.074       216

    accuracy                          0.654     44582
   macro avg      0.519     0.410     0.433     44582
weighted avg      0.643     0.654     0.644     44582

